# Lab 3: ระบบ RAG Part I
RAG หรือ Retrieval-Augmented Generation คือ เทคนิคการเพิ่มความสามารถให้ LLMs โดยการดึงข้อมูลจากแหล่งข้อมูลภายนอกมาใช้ประกอบการสร้างคำตอบ แทนที่จะพึ่งพาเพียงความรู้ที่โมเดลมีอยู่เดิมจากการเทรน เพื่อลดการ Hallucination และทำให้สามารถตอบคำถามเกี่ยวกับข้อมูลเฉพาะชุดนั้นๆ ได้อย่างถูกต้องมากขึ้น


## เป้าหมาย: 

พัฒนาสร้างระบบ RAG (Retrieval-Augmented Generation) เป็นระบบแนะนำกระทู้ Pantip ที่เกี่ยวข้อง โดยอ้างอิงข้อมูล [krathu-500
](https://github.com/Pittawat2542/krathu-500) 

โดยใช้ `InMemoryVectorStore` และเปรียบเทียบ Retrieval 3 รูปแบบ

1. **Simple Retrieval** — Dense semantic search
2. **Hybrid Retrieval** — Dense retrieval + BM25 และรวมอันดับด้วย Reciprocal Rank Fusion (RRF)
3. **Retrieval + Reranking** — Dense retrieval แบบกว้าง แล้วให้ LLM จัดอันดับใหม่ก่อนตอบ



## หัวข้อใน Lab 3: 

#### Part I: Indexing 

ขั้นตอนนี้ คือ การเตรียมข้อมูลเพื่อให้ AI ใช้ในการค้นหาข้อมูล

1. **Load** — อ่านข้อมูลจากไฟล์เอกสาร
2. **Chunk** — แบ่งเอกสารเป็นชิ้นย่อยๆ
3. **Embed** — แปลงแต่ละชิ้นเป็นเวกเตอร์
4. **Index (Vector Store)** — เก็บเวกเตอร์ไว้ในฐานข้อมูลเวกเตอร์

#### Part II: Generating 
_... ติดตามใน Lab3.2_

![RAG Pipeline](https://docs.nvidia.com/nemo-framework/user-guide/24.12/_images/rag_pipeline.png "RAG Pipeline")

credit: https://docs.nvidia.com/nemo-framework/user-guide/24.12/rag/ragoverview.html

## ภาพรวมสถาปัตยกรรม

```mermaid
flowchart TB
  CSV["Load: posts.csv + comments.csv"] --> DOC["Chunks"]
  style CSV fill:#ff9999,stroke:#333
  
  DOC --> DENSE["InMemoryVectorStore"]
  DOC --> BM25["BM25 index"]
```

## Step 1: Load — อ่านข้อมูลจากไฟล์เอกสาร

เริ่มต้นจาก download ข้อมูลจาก https://github.com/Pittawat2542/krathu-500

`git clone https://github.com/Pittawat2542/krathu-500.git`


โดยใน Lab นี้ เราจะกำหนดให้ **หนึ่ง `Document` แทนหนึ่งกระทู้** โดยประกอบด้วย
* ชื่อกระทู้
* เนื้อหาต้นกระทู้
* และความคิดเห็นทั้งหมด 

In [182]:
ls krathu-500

README.md                example/                 posts.csv
baseline-model/          labeled/                 requirements.txt
chromedriver*            main.py                  small-dataset-generator/
comments.csv             post-processing/


In [183]:
import pandas as pd

posts_df = pd.read_csv("./krathu-500/post-processing/posts.csv", dtype={"id": "string"})
comments_df = pd.read_csv(
    "./krathu-500/post-processing/comments.csv",
    dtype={
        "comment_id": "string",
        "reply_to": "string",
        "text": "string",
    },
)

In [196]:
# จำกัดจำนวน Posts ที่สนใจ แค่ 100 ข้อมูลเท่านั้น
posts_df = posts_df.head(100)

In [197]:
posts_df.head()

,id,title,url,comment_count,vote_count,published_at
0,38597354,ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงห...,https://pantip.com/topic/38597354,126,3,2019-02-26 23:06:26+06:42
1,41041562,เด็กจบใหม่กับความเครียดจากครอบครัวและการหางาน,https://pantip.com/topic/41041562,211,0,2021-10-15 01:21:22+06:42
2,41040956,พนักงานออฟฟิศ สิ่งไหนที่เจ้านายทำแล้ว พนักงานร...,https://pantip.com/topic/41040956,275,0,2021-10-14 19:51:54+06:42
3,41031597,มีวิธีรับมืออย่างไรเมื่อเจอกับคนที่ทำงานเก่งมา...,https://pantip.com/topic/41031597,496,15,2021-10-10 09:05:21+06:42
4,30467953,คนขายประกันที่บอกว่าไปเที่ยวเมืองนอก ได้เงินเด...,https://pantip.com/topic/30467953,538,0,2013-05-10 09:59:20+06:42


In [198]:
comments_df.head()

,comment_id,text,collected_at,published_at,reply_to,post_id
0,38597354-0,ผมคิดถึงเรื่องนี้มาหลายครั้ง แต่ก็ไม่เชิงว่าค...,2021-10-19 16:42:41.984684,2019-02-26 23:06:26+06:42,<NA>,38597354
1,38597354-7817b4e7-6c30-414a-8145-c96864d38409,ซื้อความรักไม่ได้ ถึงเปย์ก็ได้แต่คนไม่จริงใจ,2021-10-19 16:42:42.034824,2019-02-26 23:09:18+06:42,38597354.0,38597354
2,38597354-314fe8b7-2ded-4b50-990b-21bded1edbf5,อยากได้ ไม่ได้ทุกข์ พอได้แล้วสุข สุขแล้วเบื่อ\...,2021-10-19 16:42:42.064554,2019-02-27 00:00:36+06:42,38597354.0,38597354
3,38597354-4f1ae8a3-3eac-46f3-91de-eca33be44e94,ความสุขมันก็คือการได้รับในสิ่งที่ตัวเองต้องการ...,2021-10-19 16:42:42.096689,2019-02-27 02:31:00+06:42,38597354.0,38597354
4,38597354-38e7901d-9402-445d-a235-99d2e772c641,ความสุขของแต่ละคนมันอยู่ที่นิยามของเจ้าตัวครับ,2021-10-19 16:42:42.125382,2019-02-27 04:34:28+06:42,38597354.0,38597354


In [199]:
comments_df["post_id"] = comments_df["comment_id"].str.extract(r"^(\d+)-", expand=False)

In [200]:
print(f"posts.csv: {len(posts_df):,} records")
print(f"comments.csv: {len(comments_df):,} records")

posts.csv: 100 records
comments.csv: 63,867 records


In [201]:
from langchain_core.documents import Document

post_lookup = posts_df.set_index("id").to_dict(orient="index")
source_documents: list[Document] = []

for post_id, comments_group in comments_df.groupby("post_id", sort=False):
    if pd.isna(post_id) or post_id not in post_lookup:
        continue

    post = post_lookup[post_id]
    content_parts: list[str] = []

    for row in comments_group.itertuples(index=False):
        text = row.text
        if pd.isna(text) or (text is None) or str(text).strip()=="":
            continue

        is_post = str(row.comment_id).endswith("-0")
        source_type = "post" if is_post else "comment"
        label = "เนื้อหากระทู้" if is_post else "ความคิดเห็น"

        content_parts.append(
            f"ประเภทข้อมูล: {label}\n"
            f"รหัสข้อมูล: {row.comment_id}\n"
            f"เผยแพร่เมื่อ: {row.published_at}\n"
            f"ตอบกลับ: {str(row.reply_to)}\n"
            f"เนื้อหา:\n{text}"
        )

    if not content_parts:
        continue

    page_content = (
        f"ชื่อกระทู้: {post['title']}\n"
        f"URL: {post['url']}\n\n"
        + "\n\n---\n\n".join(content_parts)
    )

    if len(page_content) > 150000:
        continue

    source_documents.append(
        Document(
            page_content=page_content,
            metadata={
                "document_id": str(post_id),
                "post_id": str(post_id),
                "post_title": str(post["title"]),
                "source_type": "post_with_comments",
                "url": str(post["url"]),
                "content_count": len(content_parts),
                "comment_count": sum(
                    not str(comment_id).endswith("-0")
                    for comment_id in comments_group["comment_id"]
                ),
            },
        )
    )

print(f"source documents: {len(source_documents):,}")

source documents: 95


In [202]:
print(source_documents[0].page_content[0:500]+"...")

ชื่อกระทู้: ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงหรือ ที่เงินมันซื้อความสุขได้
URL: https://pantip.com/topic/38597354

ประเภทข้อมูล: เนื้อหากระทู้
รหัสข้อมูล: 38597354-0
เผยแพร่เมื่อ: 2019-02-26 23:06:26+06:42
ตอบกลับ: <NA>
เนื้อหา:
ผมคิดถึงเรื่องนี้มาหลายครั้ง  แต่ก็ไม่เชิงว่าครุ่นคิดอยู่ตลอดเวลา
แต่อารมณ์มันแบบ  แวบขึ้นมาในหัวเป็นพักๆ   เวลาที่รู้สึกว่าตัวเองไม่มีความสุขมาก อย่างที่ควรจะเป็น

เลยอยากรู้ว่า  มีใครเป็นเหมือนกันบ้าง   ที่พอเราเดินมาถึงจุดๆหนึ่งในชีวิตในแบบที่เราอยากจะเป็น...


In [203]:
document_ids = [doc.metadata["document_id"] for doc in source_documents]

document_summary = pd.DataFrame(
    {
        "document_id": document_ids,
        "characters": [len(doc.page_content) for doc in source_documents],
    }
)
print(f"documents: {len(source_documents):,}")
document_summary.describe(include="all")


documents: 95


,document_id,characters
count,95,95.000000
unique,95,NaN
top,38597354,NaN
freq,1,NaN
mean,NaN,51001.800000
std,NaN,24445.657951
min,NaN,19189.000000
25%,NaN,31678.000000
50%,NaN,45361.000000
75%,NaN,63819.500000


## Step 2: Chunk — แบ่งเอกสารเป็นชิ้นย่อยๆ

เอกสารยาวจะถูกแบ่งเป็น chunks ขนาดใกล้เคียงกัน โดยมี overlap เพื่อไม่ให้ใจความบริเวณรอยต่อขาดหาย

ทั้งนี้ เทคนิคการแบ่งมีได้หลากหลายรูปแบบ เช่น

* **Fixed-size Chunking**: แบ่งตามจำนวนตัวอักษรหรือจำนวน token ที่กำหนด เป็นวิธีที่เรียบง่ายและประมวลผลได้รวดเร็ว แต่อาจตัดข้อความกลางประโยคหรือกลางประเด็น

* **Recursive Chunking**: แบ่งข้อความตามลำดับของโครงสร้าง เช่น ย่อหน้า บรรทัด ประโยค และคำ หากส่วนใดยังมีขนาดใหญ่เกินไปจึงแบ่งซ้ำในระดับที่ละเอียดขึ้น วิธีนี้ช่วยรักษาโครงสร้างของข้อความได้ดีกว่า Fixed-size Chunking

* **Structure-based Chunking**: แบ่งตามโครงสร้างของเอกสาร เช่น หัวข้อ บท มาตรา หรือรายการ เหมาะกับเอกสารที่มีรูปแบบชัดเจน

* **Semantic Chunking**: แบ่งตามความหมายของเนื้อหา โดยตรวจหาจุดที่หัวข้อหรือบริบทเปลี่ยนไป ช่วยให้แต่ละ chunk มีใจความสมบูรณ์ แต่มีขั้นตอนและค่าใช้จ่ายในการประมวลผลมากขึ้น

* **Document-specific Chunking**: แบ่งตามหน่วยข้อมูลเฉพาะของชุดข้อมูล เช่น หนึ่งกระทู้ต่อหนึ่ง chunk หรือแยกต้นกระทู้และความคิดเห็นออกจากกัน

ทุก chunk ที่ได้จะยังคงเก็บ `metadata` ของกระทู้ต้นทาง เช่น `post_id`, `post_title` และ `url` เพื่อให้สามารถตรวจสอบแหล่งที่มาของข้อมูลและเชื่อมโยงผลการค้นคืนกลับไปยังกระทู้เดิมได้


In [204]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 5000
CHUNK_OVERLAP = 1000

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", ".", "?", "!", " ", "---"],
    add_start_index=True,
)

chunks: list[Document] = []
for source_document in source_documents:
    source_chunks = splitter.split_documents([source_document])
    for chunk_number, chunk in enumerate(source_chunks):
        if len(chunk.page_content) < 1000:
            continue

        if not chunk.page_content.startswith("ชื่อกระทู้: "):
            chunk.page_content = f"[อ้างอิงจาก ชื่อกระทู้: {chunk.metadata['post_title']}]\n...\n" + chunk.page_content
        
        chunk.metadata = {
            **chunk.metadata,
            "chunk_number": chunk_number,
            "chunk_id": (
                f"{source_document.metadata['document_id']}::"
                f"chunk-{chunk_number:03d}"
            ),
        }
        chunks.append(chunk)

In [205]:
chunk.metadata

{'document_id': '39205921',
 'post_id': '39205921',
 'post_title': 'ได้งานราชการตอนอายุ 44 ไม่มีเงินเก็บ มีภาระหนี้สินคำนวณแล้วต้องติดลบ แต่อนาคตมั่นคงควรทำมั้ย',
 'source_type': 'post_with_comments',
 'url': 'https://pantip.com/topic/39205921',
 'content_count': 109,
 'comment_count': 108,
 'start_index': 48895,
 'chunk_number': 12,
 'chunk_id': '39205921::chunk-012'}

In [206]:
for chunk in chunks[0:2]:
    print("====", chunk.metadata["chunk_id"], "====")
    print(chunk.page_content[0:500]+"...")
    print("======== END ========")
    print()
    

==== 38597354::chunk-000 ====
ชื่อกระทู้: ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงหรือ ที่เงินมันซื้อความสุขได้
URL: https://pantip.com/topic/38597354

ประเภทข้อมูล: เนื้อหากระทู้
รหัสข้อมูล: 38597354-0
เผยแพร่เมื่อ: 2019-02-26 23:06:26+06:42
ตอบกลับ: <NA>
เนื้อหา:
ผมคิดถึงเรื่องนี้มาหลายครั้ง  แต่ก็ไม่เชิงว่าครุ่นคิดอยู่ตลอดเวลา
แต่อารมณ์มันแบบ  แวบขึ้นมาในหัวเป็นพักๆ   เวลาที่รู้สึกว่าตัวเองไม่มีความสุขมาก อย่างที่ควรจะเป็น

เลยอยากรู้ว่า  มีใครเป็นเหมือนกันบ้าง   ที่พอเราเดินมาถึงจุดๆหนึ่งในชีวิตในแบบที่เราอยากจะเป็น...
======== END ========

==== 38597354::chunk-001 ====
[อ้างอิงจาก ชื่อกระทู้: ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงหรือ ที่เงินมันซื้อความสุขได้]
...
ความสุขในรูปแบบนามธรรม กับ วัตถุ มันต่างกันมาก

---

ประเภทข้อมูล: ความคิดเห็น
รหัสข้อมูล: 38597354-4f1ae8a3-3eac-46f3-91de-eca33be44e94
เผยแพร่เมื่อ: 2019-02-27 02:31:00+06:42
ตอบกลับ: 38597354.0
เนื้อหา:
ความสุขมันก็คือการได้รับในสิ่งที่ตัวเองต้องการน่ะแหละครับ คนที่มีความสุขที่สุดในโลกคือคนที่มองหาสิ่งที่

### เทียบข้อมูล Chunks vs Documents

In [207]:
print(f"documents: {len(source_documents):,}")
document_summary.describe(include="all")


documents: 95


,document_id,characters
count,95,95.000000
unique,95,NaN
top,38597354,NaN
freq,1,NaN
mean,NaN,51001.800000
std,NaN,24445.657951
min,NaN,19189.000000
25%,NaN,31678.000000
50%,NaN,45361.000000
75%,NaN,63819.500000


In [208]:
chunk_ids = [doc.metadata["chunk_id"] for doc in chunks]

chunk_summary = pd.DataFrame(
    {
        "chunk_id": chunk_ids,
        "source_type": [doc.metadata["source_type"] for doc in chunks],
        "characters": [len(doc.page_content) for doc in chunks],
    }
)
print(f"chunks: {len(chunks):,}")
chunk_summary.describe(include="all")


chunks: 1,230


,chunk_id,source_type,characters
count,1230,1230,1230.000000
unique,1230,1,NaN
top,38597354::chunk-000,post_with_comments,NaN
freq,1,1230,NaN
mean,NaN,NaN,4763.176423
std,NaN,NaN,556.497866
min,NaN,NaN,1280.000000
25%,NaN,NaN,4781.000000
50%,NaN,NaN,4936.000000
75%,NaN,NaN,5015.750000


## (Optional) แก้ไข Step 1+2 

เนื่องจาก เดิมกำหนดให้ `1 Document` แทน `1 Post` โดยรวมเนื้อหากระทู้และความคิดเห็นทั้งหมดไว้ในเอกสารเดียวกัน 

จะสังเกตว่า กระทู้แต่ละก้อนมีเนื้อหาและความคิดเห็นจำนวนมาก เมื่อนำเอกสารไปแบ่งเป็น `chunks` ทำให้ความคิดเห็นบางส่วนถูกตัดออกจากกัน หรือข้อความจากหลายความคิดเห็นถูกรวมอยู่ใน chunk เดียวกัน ส่งผลให้บริบทและความสัมพันธ์ระหว่างข้อความไม่ชัดเจน และอาจลดความแม่นยำในการค้นคืนข้อมูล

ดังนั้น ในส่วนนี้นี้จึงปรับโครงสร้างข้อมูลเป็น **`1 Document` แทน `1 Comment`** โดยถือว่าเนื้อหาต้นกระทู้เป็น comment แรกของกระทู้ด้วย วิธีนี้ช่วยรักษาขอบเขตและความหมายของแต่ละข้อความ ทำให้การสร้าง embeddings และการค้นคืนข้อมูลดำเนินการในระดับความคิดเห็นได้อย่างเหมาะสมยิ่งขึ้น


_NOTE: สังเกตได้ว่า ลักษณะของข้อมูล มีผลกับการออกแบบระบบ RAG มาก_


In [209]:
# post_lookup = posts_df.set_index("id").to_dict(orient="index")
# content_post_ids = set(comments_df["post_id"].dropna())
# source_documents: list[Document] = []

# for row in comments_df.itertuples(index=False):
#     text = row.text
#     if pd.isna(text) or (text is None) or str(text).strip()=="":
#             continue

#     post = post_lookup[row.post_id]
#     source_type = "post" if str(row.comment_id).endswith("-0") else "comment"
#     label = "เนื้อหากระทู้" if source_type == "post" else "ความคิดเห็น"
#     page_content = (
#         f"ชื่อกระทู้: {post['title']}\n"
#         f"ประเภทข้อมูล: {label}\n"
#         f"เนื้อหา:\n{text}"
#     )

#     if len(page_content) < 200 or len(page_content) > 5000:
#         continue

#     source_documents.append(
#         Document(
#             page_content=page_content,
#             metadata={
#                 "document_id": str(row.comment_id),
#                 "post_id": str(row.post_id),
#                 "post_title": str(post["title"]),
#                 "source_type": source_type,
#                 "url": str(post["url"]),
#                 "published_at": row.published_at,
#                 "reply_to": str(row.reply_to),
#             },
#         )
#     )
    
# print(f"source documents: {len(source_documents):,}")


In [210]:
# print(source_documents[2].page_content)

In [211]:
# document_ids = [doc.metadata["document_id"] for doc in source_documents]

# document_summary = pd.DataFrame(
#     {
#         "document_id": document_ids,
#         "characters": [len(doc.page_content) for doc in source_documents],
#     }
# )
# print(f"documents: {len(source_documents):,}")
# document_summary.describe(include="all")


In [212]:
# from langchain_text_splitters import RecursiveCharacterTextSplitter

# CHUNK_SIZE = 1500
# CHUNK_OVERLAP = 200

# splitter = RecursiveCharacterTextSplitter(
#     chunk_size=CHUNK_SIZE,
#     chunk_overlap=CHUNK_OVERLAP,
#     length_function=len,
#     separators=["\n\n", "\n", ".", "?", "!", " ", ""],
#     add_start_index=True,
# )

# chunks: list[Document] = []
# for source_document in source_documents:
#     source_chunks = splitter.split_documents([source_document])
#     for chunk_number, chunk in enumerate(source_chunks):
#         if len(chunk.page_content) <= 100:
#             continue
            
#         chunk.metadata = {
#             **chunk.metadata,
#             "chunk_number": chunk_number,
#             "chunk_id": (
#                 f"{source_document.metadata['document_id']}::"
#                 f"chunk-{chunk_number:03d}"
#             ),
#         }
#         chunks.append(chunk)

In [213]:
# document_ids = [doc.metadata["document_id"] for doc in source_documents]

# document_summary = pd.DataFrame(
#     {
#         "document_id": document_ids,
#         "characters": [len(doc.page_content) for doc in source_documents],
#     }
# )
# print(f"documents: {len(source_documents):,}")
# document_summary.describe(include="all")


In [214]:
# for chunk in chunks[0:4]:
#     print("====", chunk.metadata["chunk_id"], "====")
#     print(chunk.page_content)
#     print()
    

In [215]:
# chunk_ids = [doc.metadata["chunk_id"] for doc in chunks]

# chunk_summary = pd.DataFrame(
#     {
#         "chunk_id": chunk_ids,
#         "source_type": [doc.metadata["source_type"] for doc in chunks],
#         "characters": [len(doc.page_content) for doc in chunks],
#     }
# )
# print(f"chunks: {len(chunks):,}")
# chunk_summary.describe(include="all")


## Step 3: Embed — แปลงแต่ละ chunk เป็นเวกเตอร์

ในขั้นตอนนี้ เราจะใช้ Embedding Model แปลง `page_content` ของแต่ละ chunk ให้เป็น Dense Vector

In [216]:
import os
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
MODEL_NAME = "gemini-embedding-2"
OUTPUT_DIMENSION = 768

embedder = GoogleGenerativeAIEmbeddings(
    api_key=GEMINI_KEY,
    model=MODEL_NAME,
    task_type="SEMANTIC_SIMILARITY",
    output_dimensionality=OUTPUT_DIMENSION,
)

In [225]:
run_embedding = False

In [226]:
import time
from tqdm import tqdm

if run_embedding:
    chunk_texts = [chunk.page_content for chunk in chunks]
    
    embedding_started = time.perf_counter()
    # # without batch
    # chunk_vectors = embedder.embed_documents(chunk_texts)
    
    # with batch
    BATCH_SIZE = 100
    chunk_vectors = []
    for i in tqdm(range(0, len(chunk_texts), BATCH_SIZE), desc="Generating Embeddings"):
        batch = chunk_texts[i : i + BATCH_SIZE]
        batch_embeddings = embedder.embed_documents(batch)
        chunk_vectors.extend(batch_embeddings)
    
    
    embedding_seconds = time.perf_counter() - embedding_started
    
    print(f"Embedded {len(chunk_vectors):,} chunks")
    print(f"Vector dimension: {len(chunk_vectors[0]):,}")
    print(f"Embedding time: {embedding_seconds:.2f} seconds")
    
# Embedded 1,230 chunks
# Vector dimension: 768
# Embedding time: 590.64 seconds

In [227]:
import pickle
from pathlib import Path

if run_embedding:
    output_path = Path("./Examples/documents_dense_vectors.pkl")
    
    index_data = {
        "chunks": chunks,
        "chunk_ids": chunk_ids,
        "chunk_vectors": chunk_vectors,
        "model_name": MODEL_NAME,
        "output_dimension": OUTPUT_DIMENSION,
    }
    
    with output_path.open("wb") as file:
        pickle.dump(index_data, file)
    
    print(f"Saved to: {output_path}")
    print(f"Vectors: {len(chunk_vectors):,}")

### Load pre-computed embeddings

In [228]:
input_path = Path("./Examples/documents_dense_vectors.pkl")
with input_path.open("rb") as file:
    index_data = pickle.load(file)

chunks = index_data["chunks"]
chunk_ids = index_data["chunk_ids"]
chunk_vectors = index_data["chunk_vectors"]

MODEL_NAME = index_data["model_name"]
OUTPUT_DIMENSION = index_data["output_dimension"]

print(f"Loaded {len(chunk_vectors):,} vectors")
print(f"Model: {MODEL_NAME}")
print(f"Vector dimension: {OUTPUT_DIMENSION}")

Loaded 1,230 vectors
Model: gemini-embedding-2
Vector dimension: 768


## Step 4: Index — เก็บเวกเตอร์ไว้ค้นหา


#### Dense Vector Store
นำ vectors ที่สร้างไว้แล้ว พร้อมข้อความและ metadata ไปจัดเก็บใน InMemoryVectorStore

ใน application จริงๆ มักใช้ **Vector Database** เช่น Qdrant, Pinecone แต่เพื่อความง่าย แล็บนี้จะเก็บเวกเตอร์ไว้ใน `InMemoryVectorStore` ซึ่งเก็บ vectors ในหน่วยความจำของ Python process

ข้อควรทราบ:
- เมื่อ restart runtime ต้องสร้าง index ใหม่
- ไม่เหมาะกับ production corpus ขนาดใหญ่หรือระบบที่ต้องมี persistence/concurrent access


In [229]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embedder)

indexing_started = time.perf_counter()
for chunk, chunk_id, vector in zip(
    chunks,
    chunk_ids,
    chunk_vectors,
    strict=True,
):
    vector_store.store[str(chunk_id)] = {
        "id": str(chunk_id),
        "vector": vector,
        "text": chunk.page_content,
        "metadata": chunk.metadata,
    }

indexing_seconds = time.perf_counter() - indexing_started

print(f"Indexed {len(vector_store.store):,} chunks")
print(f"Indexing time: {indexing_seconds:.4f} seconds")

Indexed 1,230 chunks
Indexing time: 0.0050 seconds


#### Test Dense Vector Store

In [275]:
test_query = "เงินซื้อความสุขได้จริงหรือไม่"
dense_preview = vector_store.similarity_search_with_score(test_query, k=3)

for doc, score in dense_preview:
    print("Id", doc.id)
    print("Similarity Score:", score)
    print("Content:")
    print(doc.page_content[0:500]+"...")
    print("=="*25)

Id 35725482::chunk-019
Similarity Score: 0.5964780438085425
Content:
[อ้างอิงจาก ชื่อกระทู้: อยากทราบอาชีพ ของคนที่มีรายได้ 80k+(80,000+) ขึ้นไปอ่ะครับ]
...
---

ประเภทข้อมูล: ความคิดเห็น
รหัสข้อมูล: 35725482-728a6050-44e2-4301-b422-d94a5f781f69
เผยแพร่เมื่อ: 2021-04-18 14:37:16+06:42
ตอบกลับ: 35725482.0
เนื้อหา:
คนรวยจำนวนมากไม่ได้มีวุฒิการศึกษาดีครับ แต่เป็นเจ้านายของคนวุฒิการศึกษาดี ดูคนรวยตามต่างจังหวัดเป็นตัวอย่างครับ ทำสิ่งที่เข้าใจง่าย มีความต้องการสูง ไม่ต้องเป็นแบรนด์หรูก็รวยได้ สำคัญว่าต้องใช้เงินขยายต่อไปเรื่อยๆ บางคนเล่นหุ้น เล่นบิทคับ ซื้อที่ขายต่อ แต่เฮียแถวบ้านเริ่มจากเปิดร้านวัสดุอุปกรณ์ ขยายเป็นเครื่องใช้ไฟฟ้า ปัจจุบันนี้ขายแบบเดียวกับโฮมโปร ครองขายในตัวจังหวัด แล้วมีจังหวัดใกล้เคียงมาซื้อเพราะหาของได้ครบอีก รวยไม่หยุดเลยทีนี้

---

ประเภทข้อมูล: ความคิดเห็น
รหัสข้อมูล: 35725482-f233cc4b-cb22-4a8e-8022-e2ee32f83d80
เผยแพร่เมื่อ: 2021-04-19 14:38:32+06:42
ตอบกลับ: 35725482.0
เนื้อหา:
เงินเดือน 80,000 ระดับผู้บริหารในบริษัทใหญ่เลยนะครับ ต้องอายุประมาณ 50 ขึ้นไป เก่งกว่าน

#### (Optional) Sparse Vector DB

สร้าง Sparse Vector Store ด้วยเทคนิค BM25

In [233]:
!uv pip install -q pythainlp rank-bm25

In [285]:
import unicodedata
from pythainlp.tokenize import word_tokenize
from pythainlp.util import normalize as pythainlp_th_normalize
from rank_bm25 import BM25Okapi
import numpy as np

def tokenize_thai(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFKC", text).lower()
    normalized_th = pythainlp_th_normalize(normalized)
    tokens = word_tokenize(normalized_th, engine="newmm", keep_whitespace=False)
    return [token.strip() for token in tokens if token.strip()]
    return tokens


class ThaiBM25Index:
    def __init__(self, documents: list[Document]):
        self.documents = list(documents)
        self.tokenized_corpus = []

        for document in tqdm(documents, desc="Tokenizing"):
            tokens = tokenize_thai(document.page_content)
            self.tokenized_corpus += [tokens]
            
        self.document_index = None

    def index(self):
        self.document_index = BM25Okapi(self.tokenized_corpus)
        
    def search(self, query: str, k: int = 4, *, include_zero_scores: bool = False):
        query_tokens = tokenize_thai(query)
        scores = self.document_index.get_scores(query_tokens)
        ranked_indices = np.argsort(scores)[::-1]

        results: list[tuple[Document, float]] = []
        for index in ranked_indices:
            score = float(scores[index])
            if score <= 0 and not include_zero_scores:
                continue
            results.append((self.documents[int(index)], score))
            if len(results) >= k:
                break
        return results

In [286]:
bm25_index = ThaiBM25Index(chunks)
bm25_index.index()
print(f"BM25 index contains {len(bm25_index.documents):,} chunks")

Tokenizing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 1230/1230 [00:11<00:00, 109.94it/s]


BM25 index contains 1,230 chunks


In [291]:
import pickle
from pathlib import Path

if run_embedding:
    output_path = Path("./Examples/documents_sparse_vectors.pkl")
    
    index_data = {
        "chunks": chunks,
        "chunk_ids": chunk_ids,
        "chunk_vectors": bm25_index,
    }
    
    with output_path.open("wb") as file:
        pickle.dump(index_data, file)
    
    print(f"Saved to: {output_path}")
    print(f"Vectors: {len(chunk_vectors):,}")

In [289]:
test_query = "เงินซื้อความสุขได้จริงหรือไม่"

bm25_preview = bm25_index.search(test_query, k=3)

for doc, score in bm25_preview:
    print("Id", doc.metadata["chunk_id"])
    print("Similarity Score:", score)
    print("Content:")
    print(doc.page_content[0:500]+"...")
    print("=="*25)


Id 38597354::chunk-002
Similarity Score: 12.759638379007969
Content:
[อ้างอิงจาก ชื่อกระทู้: ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงหรือ ที่เงินมันซื้อความสุขได้]
...
ลองหาเวลาไปเข้าวัดปฏิบัติธรรมสักหน่อยดีไหมครับ

---

ประเภทข้อมูล: ความคิดเห็น
รหัสข้อมูล: 38597354-ad030756-8f23-4bdc-a1df-1c67d7f09508
เผยแพร่เมื่อ: 2019-02-27 09:40:51+06:42
ตอบกลับ: 38597354.0
เนื้อหา:
นึกถึงตอนเป็นเด็กน้อย. แค่ได้วิ่งเล่นสนุกสนาน.  กินอิ่มนอนหลับ. ก็มีความสุขมากมาย
ทำให้ฉุกคิดได้นะว่า. ความสุขง่ายๆ. กินอิ่มนอนหลับ. ไม่มีโรคภัย. เราก็มีความสุขได้เดี๋ยวนี้. ตอนนี้เลย
แก้ไขข...
Id 38597354::chunk-007
Similarity Score: 12.723848028454764
Content:
[อ้างอิงจาก ชื่อกระทู้: ถึงคนที่มีทุกอย่างอย่างที่ฝันไว้แล้ว..ว่าจริงหรือ ที่เงินมันซื้อความสุขได้]
...
คนเรามักจะไล่ตามกิเลสโดยเสมอครับ เรามักจะคิดว่าเราต้องการอะไร ต้องการสิ่งไหน อะไรมีแล้วก็จะมากขึ้นไปอีก เป็นแบบนี้ไปไม่สิ้นสุดครับ

ในกรณีของคุณ จขกท คือมีเป้าหมายไว้ แต่ไล่ตามเป้าหมายได้ครบแล้ว ก็เลยรู้สึกโหวงๆ น่ะครับ เหมือนขาดแพชชั่นในการขับเคลื่อน

อ